In [ ]:
import pandas as pd
import numpy as np
import pysrt
import os
import subprocess
from pathlib import Path

In [ ]:
# read signbank csv
file_path = r"..\..\..\AnimData\dictionary-export.csv"
df = pd.read_csv(file_path)

df.head()

In [ ]:
def get_handshape_from_gloss(gloss):
    row = df[df['Annotation ID Gloss (Dutch)'] == gloss]
    if not row.empty:
        return row.iloc[0]['Strong Hand']
    return None

In [ ]:
def get_frames_from_timecode(timecode, fps):
    # Convert timecode to total seconds
    total_seconds = timecode.hours * 3600 + timecode.minutes * 60 + timecode.seconds + timecode.milliseconds / 1000.0
    # Calculate frame number
    frame_number = int(total_seconds * fps)
    return frame_number

In [ ]:
# read srt
test = pysrt.open(r"..\..\..\AnimData\srt\M20240925_1858_260319_0_Signbank_ID_glossen.srt")

fps = 30  # frames per second

for sub in test:
    print(get_frames_from_timecode(sub.start, fps), get_frames_from_timecode(sub.end, fps), get_handshape_from_gloss(sub.text))

In [ ]:
def get_matches_from_csv(csv_path):
    matches = []
    with open(csv_path, 'r') as f:
        next(f)  # Skip the header line
        for line in f:
            # rank, handshape_file, distance
            parts = line.strip().split(',')
            if len(parts) == 3:
                matches.append((int(parts[0]), parts[1], float(parts[2])))
    return matches

In [ ]:
# go through each fbx and compare
fbx_folder = r"..\..\fbx"

exe_loc = r"..\..\build\Release\shapeseg.exe"
repo_root = Path(exe_loc).resolve().parents[2]
exe = repo_root / "build" / "Release" / "shapeseg.exe"

segment_file = r"..\..\output\handshape_segments.csv"

matches = dict()
segments_by_fbx = dict()

for fbx_file in os.listdir(fbx_folder):
    if fbx_file.endswith(".fbx"):
        matches[fbx_file] = []

        fbx = repo_root / "fbx" / fbx_file

        # Update output files by running the command
        result = subprocess.run(
            [str(exe), "s", str(fbx)],
            cwd=repo_root,
            check=True,
            capture_output=True,
            text=True,
        )

        # Get number of segments
        file = open(segment_file, 'r')
        segments = [(line.split(',')[0], line.split(',')[1]) for line in file.readlines()[1:]]
        file.close()
        segments_by_fbx[fbx_file] = segments
        print(f"Processed {fbx_file}: {len(segments)} segments found.")

        # Get the matches for each segment
        for i in range(len(segments)):
            result = subprocess.run(
                [str(exe), "m", str(fbx), str(i)],
                cwd=repo_root,
                check=True,
                capture_output=True,
                text=True,
            )

            # get matches
            match_file = repo_root / "output" / "handshape_matches.csv"
            match_list = get_matches_from_csv(match_file)
            matches[fbx_file].append(match_list)



In [ ]:
# make a dataframe from the matches dictionary
data = []

for fbx_file, match_lists in matches.items():
    for segment_idx, match_list in enumerate(match_lists):
        for match in match_list:
            data.append({
                "fbx_file": fbx_file,
                "segment": segment_idx,
            "rank": match[0],
            "handshape_file": match[1],
            "distance": match[2]
        })
        segment_idx += 1

matches_df = pd.DataFrame(data)

matches_df.head()

In [ ]:
# Convert handshape_file to handshape
matches_df['handshape'] = matches_df['handshape_file'].str.split('/').str[-1].str.split('.').str[0]

matches_df.head()

In [ ]:
srt_folder = r"..\..\..\AnimData\srt"

def load_srt(fbx_file):
    srt_path = Path(srt_folder) / f"{Path(fbx_file).stem}_Signbank_ID_glossen.srt"
    if not srt_path.exists():
        print(f"No srt found for {fbx_file}: {srt_path}")
        return None
    print(f"Loading srt for {fbx_file}: {srt_path}")
    return pysrt.open(str(srt_path))

In [ ]:
def find_overlapping_subs_from_segment(fbx_file, segment_idx):
    srt = load_srt(fbx_file)
    if srt is None:
        return []

    # Get the start and end frames of the segment
    segment_start_frame = int(segments_by_fbx[fbx_file][segment_idx][0])
    segment_end_frame = int(segments_by_fbx[fbx_file][segment_idx][1])

    overlapping_subs = []
    for sub in srt:
        sub_start_frame = get_frames_from_timecode(sub.start, fps)
        sub_end_frame = get_frames_from_timecode(sub.end, fps)

        # Check for overlap
        if not (sub_end_frame < segment_start_frame or sub_start_frame > segment_end_frame):
            overlapping_subs.append(sub)

    if not overlapping_subs:
        print(f"No overlapping subtitles found for {fbx_file}, segment {segment_idx} (frames {segment_start_frame}-{segment_end_frame})")

    return overlapping_subs

In [ ]:
# for each unique fbx_file, segment combination, find the rank of the correct handshape
def get_correct_handshape(fbx_file, segment_idx):
    # Get overlapping subs
    overlapping_subs = find_overlapping_subs_from_segment(fbx_file, segment_idx)

    if not overlapping_subs:
        return []

    # Get the correct handshape from the overlapping subs
    contender_handshapes = []
    for sub in overlapping_subs:
        contender_handshapes.append(get_handshape_from_gloss(sub.text))

    # Remove None values
    contender_handshapes = [hs for hs in contender_handshapes if hs is not None]

    if not contender_handshapes:
        print(f"No valid handshapes found for {fbx_file}, segment {segment_idx}")
        return []

    # Get the matches for this fbx_file and segment_idx
    matches_for_segment = matches_df[(matches_df['fbx_file'] == fbx_file) & (matches_df['segment'] == segment_idx)]

    if matches_for_segment.empty:
        print(f"No matches found for {fbx_file}, segment {segment_idx}")
        return []

    # Find the rank of the correct handshape
    correct_ranks = []
    for handshape in contender_handshapes:
        match_row = matches_for_segment[matches_for_segment['handshape'] == handshape]
        if not match_row.empty:
            correct_ranks.append(match_row.iloc[0]['rank'])

    return correct_ranks

In [ ]:
# Get correct ranks for all unique fbx_file, segment combinations
ranks = []

for fbx_file in matches_df['fbx_file'].unique():
    for segment_idx in matches_df[matches_df['fbx_file'] == fbx_file]['segment'].unique():
        ranks.append(get_correct_handshape(fbx_file, segment_idx))


print(ranks)